# Evaluating Zero-Shot Foundation Models for Epidemiological Forecasting of Influenza-Like Illness in Italy
**Author:** Biagio Incardona  
**Objective:** Benchmarking Foundation Models (Chronos, TimesFM, TiRex) on Italian ILI data.

---

Dear Professor,

This notebook is designed to be a fully automated, "plug-and-play" environment to reproduce the benchmarking results of my thesis. It handles the environment setup, data ingestion from the official Influcast repository, and the execution of the zero-shot foundation models.

### 💡 Pro Tip for Colab:
Before starting, please ensure you are using a **GPU Runtime**:
1. Go to **Runtime** -> **Change runtime type**.
2. Select **T4 GPU** (or better).

You can then run all cells sequentially by pressing `Ctrl + F9`.

## Step 1: Setting up the Workspace
First, we clone the project repository and install all necessary dependencies. I have included the specific versions for foundation models to ensure reproducibility.

In [ ]:
# 1. Clone the repository (Ensure this is the latest public version)
REPO_URL = "https://github.com/biagio-incardona/thesis_epiforecast_italy.git"
!git clone {REPO_URL}
%cd thesis_epiforecast_italy

# 2. Install core dependencies
# This includes foundation models (Chronos, TimesFM, TiRex) and standard ML/Stats tools.
!pip install -q -r requirements.txt

# 3. Setup Python Path to recognize our 'src' directory
import sys
import os
sys.path.append(os.getcwd())

### 📤 Optional: Upload Local Code Changes
If you have made local changes to `src/` or `scripts/` and want to test them without pushing to GitHub, run the cell below. 

1. Create a zip of your local folders: `zip -r local_code.zip src scripts` 
2. Run the cell below and upload `local_code.zip` when prompted.

In [ ]:
from google.colab import files
import shutil

uploaded = files.upload()
if 'local_code.zip' in uploaded:
    print("Unpacking local code updates...")
    !unzip -o local_code.zip
    print("Done! Your local src/ and scripts/ are now active.")

## Step 2: Automated Data Ingestion
To ensure we are working with the most up-to-date surveillance data, this step pulls raw ILI counts from the official Influcast GitHub and transforms them into the standardized format used for the benchmark.

In [ ]:
print("Ingesting raw ILI data from Influcast...")
!python3 src/data/ingestion.py

print("\nPreprocessing data into standardized format...")
!python3 src/data/preprocessing.py

# Verify data readiness
if os.path.exists('data/processed/ili_gold.csv'):
    import pandas as pd
    df = pd.read_csv('data/processed/ili_gold.csv')
    print(f"\nSUCCESS: Data ready. Rows: {len(df)}. Last date: {df['ds'].max()}")
else:
    print("\nERROR: Data preparation failed. Please check the ingestion logs.")

## Step 3: Running the Full Benchmarking Suite
Now we run the core experiment. To make this a fair academic comparison, we evaluate both **Classical Baselines**, **ML Baselines** and the new **Foundation Models**.

To ensure stability in the Colab environment, we have split the benchmarking into three stages.

In [ ]:
print(">>> 1. Running Classical Baselines (SARIMA, ARIMA, Prophet, etc.)...")
!python3 scripts/benchmark_ili_national.py --model Naive
!python3 scripts/benchmark_ili_national.py --model SeasonalNaive --append
!python3 scripts/benchmark_ili_national.py --model ETS --append
!python3 scripts/benchmark_ili_national.py --model ARIMA --append
!python3 scripts/benchmark_ili_national.py --model SARIMA --append
!python3 scripts/benchmark_ili_national.py --model Prophet --append

In [ ]:
print(">>> 2. Running ML Baselines (LightGBM, XGBoost)...")
!python3 scripts/benchmark_ili_national.py --model LightGBM --tune --append
!python3 scripts/benchmark_ili_national.py --model XGBoost --tune --append

In [ ]:
import torch
if not torch.cuda.is_available():
    print("WARNING: GPU not detected. Foundation models will be extremely slow on CPU.")

print(">>> 3. Running Foundation Models (Chronos, TimesFM, TiRex)...")
!python3 scripts/benchmark_ili_national.py --model Chronos --model-size small --append
!python3 scripts/benchmark_ili_national.py --model Chronos --model-size large --append
!python3 scripts/benchmark_ili_national.py --model TimesFM --append
!python3 scripts/benchmark_ili_national.py --model TiRex --append

print("\n*Note: TimeGPT is omitted due to enterprise API key requirements discussed in our email exchange.*")

## Step 4: Visualizing the Results
Instead of just looking at CSVs, let's visualize how these foundation models performed. 

I have prepared four views for you:
1. **Performance vs Horizon**: How error scales as we forecast further into the future.
2. **Error Heatmaps**: A side-by-side look at all academic metrics. *(Note: For `Coverage95`, the optimal target is `0.95`. Values closer to `1.0` indicate over-calibration/too wide intervals.)*
3. **Forecast Trajectories**: A look at the actual Italian ILI curve versus our model predictions.
4. **Peak Analysis**: A specialized view of how models captured seasonal peaks.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import os
from src.evaluation.metrics import evaluate_forecasts
from src.evaluation.peak_metrics import calculate_seasonal_peak_metrics

forecasts_path = 'results/phase2/all_models_forecasts.csv'
truth_path = 'data/processed/ili_gold.csv'

if os.path.exists(forecasts_path) and os.path.exists(truth_path):
    forecasts_df = pd.read_csv(forecasts_path)
    truth_df = pd.read_csv(truth_path)
    
    # Convert dates and filter truth
    forecasts_df['target_date'] = pd.to_datetime(forecasts_df['target_date'])
    truth_df['ds'] = pd.to_datetime(truth_df['ds'])
    truth_df_ita = truth_df[truth_df['region'] == 'italia']

    # --- 0. Recalculate Metrics (Ensuring no leakage) ---
    print("Calculating academic metrics (MAE, RMSE, sMAPE, MASE, WIS, CRPS, Coverage)...")
    first_origin = pd.to_datetime(forecasts_df['origin'].min())
    train_slice = truth_df_ita[truth_df_ita['ds'] < first_origin].copy()
    metrics_df = evaluate_forecasts(forecasts_df, truth_df_ita, train_data=train_slice)

    print("Calculating seasonal peak metrics...")
    peak_truth = truth_df_ita.rename(columns={'ds': 'target_date', 'y': 'true_value'})
    peak_results = []
    for h in [None, 1, 2, 4, 8]:
        h_peak = calculate_seasonal_peak_metrics(forecasts_df, peak_truth, horizon=h)
        peak_results.append(h_peak)
    peak_metrics_df = pd.concat(peak_results, ignore_index=True)

    # --- 1. Metric vs Horizon (Academic Line Plots) ---
    fig, axes = plt.subplots(1, 2, figsize=(20, 7))
    sns.lineplot(data=metrics_df, x='horizon', y='MAE', hue='model', marker='o', ax=axes[0])
    axes[0].set_title('MAE vs Forecast Horizon (Lower is Better)', fontsize=14)
    axes[0].grid(True, alpha=0.3)

    sns.lineplot(data=metrics_df, x='horizon', y='WIS', hue='model', marker='s', ax=axes[1])
    axes[1].set_title('WIS vs Forecast Horizon (Probabilistic Calibration)', fontsize=14)
    axes[1].grid(True, alpha=0.3)
    plt.show()

    # --- 2. Academic Performance Heatmaps ---
    metrics_to_plot = ['MASE', 'sMAPE', 'WIS', 'CRPS', 'PinballLoss', 'Coverage95']
    fig, axes = plt.subplots(1, 6, figsize=(30, 6))
    axes = axes.flatten()
    for i, metric in enumerate(metrics_to_plot):
        pivot_df = metrics_df.pivot_table(index='model', columns='horizon', values=metric)
        cmap = "YlOrRd" if metric != 'Coverage95' else "viridis"
        sns.heatmap(pivot_df, annot=True, fmt=".2f", cmap=cmap, ax=axes[i])
        axes[i].set_title(f'{metric} Heatmap')
    plt.tight_layout()
    plt.show()

    # --- 3. Forecast vs Truth (Trajectory Comparison) ---
    # Select an origin that captures a seasonal peak (e.g., late 2023 or late 2024)
    peak_dates = truth_df_ita[truth_df_ita['y'] > truth_df_ita['y'].quantile(0.95)]['ds']
    potential_origins = forecasts_df[forecasts_df['target_date'].isin(peak_dates)]['origin'].unique()
    if len(potential_origins) > 0:
        # Pick the origin closest to a peak
        recent_origin = potential_origins[len(potential_origins)//2]
    else:
        recent_origin = forecasts_df['origin'].unique()[-5]
        
    subset = forecasts_df[forecasts_df['origin'] == recent_origin].copy()
    plt.figure(figsize=(15, 7))
    relevant_truth = truth_df_ita[(truth_df_ita['ds'] >= pd.to_datetime(recent_origin) - pd.Timedelta(weeks=12)) & 
                                  (truth_df_ita['ds'] <= subset['target_date'].max() + pd.Timedelta(weeks=4))]
    plt.plot(relevant_truth['ds'], relevant_truth['y'], 'k-o', label='Actual ILI', linewidth=3, zorder=10)
    
    # Plot top 3 models + SARIMA + Seasonal Naive for clarity
    top_3_models = metrics_df.groupby('model')['MAE'].mean().nsmallest(3).index.tolist()
    models_to_plot = list(set(top_3_models + ['SARIMA', 'SeasonalNaive']))
    
    for model in models_to_plot:
        if model in subset['model'].unique():
            m_data = subset[subset['model'] == model]
            median = m_data[np.isclose(m_data['quantile'], 0.5)].sort_values('target_date')
            plt.plot(median['target_date'], median['value'], '--', label=model, linewidth=2)

    plt.axvline(pd.to_datetime(recent_origin), color='red', linestyle=':', label='Forecast Origin')
    plt.title(f'National Forecast Comparison (Capture Peak: Origin {recent_origin})', fontsize=16)
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.show()

    # --- 4. Seasonal Peak Errors ---
    if not peak_metrics_df.empty:
        fig, axes = plt.subplots(1, 2, figsize=(20, 7))
        
        # View A: Timing vs Intensity Scatter
        plot_peaks = peak_metrics_df[peak_metrics_df['horizon'] == 'latest']
        sns.scatterplot(data=plot_peaks, x='peak_timing_error_weeks', y='peak_intensity_error_rel', hue='model', s=200, alpha=0.7, ax=axes[0])
        axes[0].axhline(0, color='black', linestyle='--')
        axes[0].axvline(0, color='black', linestyle='--')
        axes[0].set_title("Peak Performance: Timing vs Intensity", fontsize=15)
        axes[0].grid(True, alpha=0.3)
        axes[0].set_xlabel("Timing Error (Weeks) - Negative is early")
        axes[0].set_ylabel("Relative Intensity Error (Ratio)")

        # View B: Probability Coverage (Calibration Curve)
        print("\nGenerating Probability Coverage (Calibration) plot...")
        eval_merged = forecasts_df.merge(truth_df_ita[['ds', 'y']], left_on='target_date', right_on='ds', how='inner')
        
        for model in forecasts_df['model'].unique():
            m_data = eval_merged[eval_merged['model'] == model]
            if m_data.empty: continue
            
            empirical_cov = []
            theoretical_qs = sorted(m_data['quantile'].unique())
            for q in theoretical_qs:
                q_data = m_data[np.isclose(m_data['quantile'], q)]
                coverage = (q_data['y'] <= q_data['value']).mean()
                empirical_cov.append(coverage)
            
            axes[1].plot(theoretical_qs, empirical_cov, marker='.', label=model, alpha=0.8)
        
        axes[1].plot([0, 1], [0, 1], 'k--', label='Perfect Calibration')
        axes[1].set_title("Probability Coverage (Calibration Curve)", fontsize=15)
        axes[1].set_xlabel("Theoretical Quantile")
        axes[1].set_ylabel("Empirical Coverage")
        axes[1].legend(bbox_to_anchor=(1.05, 1), loc='upper left')
        axes[1].grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.show()

    # --- 5. Summary Table & Leaderboard ---
    print("\n" + "="*60)
    print("        ACADEMIC PERFORMANCE SUMMARY (Averaged Across Horizons)")
    print("="*60)
    
    # Aggregate standard metrics
    metric_cols = ['MAE', 'RMSE', 'sMAPE', 'MASE', 'WIS', 'CRPS', 'PinballLoss', 'Coverage95']
    summary = metrics_df.groupby('model')[metric_cols].mean()
    
    # Integrate Peak Metrics
    if not peak_metrics_df.empty:
        # Calculate Absolute errors for averaging across seasons/horizons
        peak_metrics_df['abs_timing_err'] = peak_metrics_df['peak_timing_error_weeks'].abs()
        peak_metrics_df['abs_intensity_err'] = peak_metrics_df['peak_intensity_error_rel'].abs()
        
        peak_summary = peak_metrics_df.groupby('model').agg({
            'abs_timing_err': 'mean',
            'abs_intensity_err': 'mean'
        }).rename(columns={
            'abs_timing_err': 'Peak Timing Err (wks)',
            'abs_intensity_err': 'Peak Intens. Err (%)'
        })
        # Convert intensity to percentage for readability
        peak_summary['Peak Intens. Err (%)'] *= 100
        
        summary = summary.join(peak_summary)
        metric_cols += ['Peak Timing Err (wks)', 'Peak Intens. Err (%)']
    
    summary = summary.sort_values('MAE')
    
    # Calculate improvement over Seasonal Naive
    snaive_mae = summary.loc['SeasonalNaive', 'MAE'] if 'SeasonalNaive' in summary.index else summary['MAE'].max()
    summary['Skill vs S.Naive (%)'] = (1 - (summary['MAE'] / snaive_mae)) * 100
    
    # Calculate Leaderboard (Average Rank across all metrics)
    # For Coverage95, higher is better, so we rank descending
    ranks = summary.copy()
    for col in metric_cols:
        ascending = False if col == 'Coverage95' else True
        ranks[col] = summary[col].rank(ascending=ascending)
    
    summary['Avg Rank'] = ranks[metric_cols].mean(axis=1)
    summary = summary.sort_values('Avg Rank')

    from IPython.display import display
    display(summary.style.background_gradient(cmap='RdYlGn_r', subset=['MAE', 'RMSE', 'sMAPE', 'MASE', 'WIS', 'CRPS', 'PinballLoss', 'Avg Rank', 'Peak Timing Err (wks)', 'Peak Intens. Err (%)'])
                  .background_gradient(cmap='RdYlGn', subset=['Coverage95', 'Skill vs S.Naive (%)']))
    
    print("\n🏆 LEADERBOARD INSIGHT: Models are ranked by their 'Average Rank' across all academic metrics (including Peaks).")
else:
    print("Results not found. Please ensure Step 3 completed successfully.")

## Step 5: Insights & Strategic Conclusion
This benchmark represents a rigorous evaluation of Zero-Shot Foundation Models against classical and ML baselines for Italian ILI data. 

### 🏆 The "Skill over Naive" Test
In epidemiological forecasting, the **Seasonal Naive** baseline is notoriously hard to beat. Our results show that:
- **Chronos-Large** and **TimesFM** demonstrate significant "skill" (improvement over naive), particularly at longer horizons (4-8 weeks).
- Classical models like **SARIMA** excel at short-term accuracy but often fail to capture the magnitude of extreme peaks compared to foundation models.

### 🏔️ Peak Timing & Intensity Analysis
One of the most critical requirements for public health planning is the accurate prediction of the **seasonal peak**. Our analysis reveals:
1. **Intensity Calibration:** Foundation models (TSFMs) generally show a lower relative intensity error than statistical baselines. While SARIMA often under-predicts the intensity of high-surge seasons (viewing them as statistical outliers), TSFMs have learned the typical "shape" of an epidemic surge from global datasets, allowing them to better anticipate the maximum burden on the healthcare system.
2. **Timing Precision:** Most models struggle with peak timing as the horizon increases. However, the **Peak Performance scatter plot** (Step 4, View 4) shows a cluster of foundation models closer to the (0,0) origin, indicating a better balance between when the peak occurs and how severe it will be. Early warning (negative timing error) is often observed in zero-shot models, which is preferable for proactive resource allocation compared to lagging indicators.

### 🛠️ Engineering the Pipeline: Technical Challenges
Moving these models from research papers to a production-ready Italian context required several engineering innovations included in this repository:

1. **Memory Management (Colab Optimization):** To fit models like `Chronos-Large` and `TimesFM` within the 16GB RAM limit of a standard T4 Colab instance, we implemented **vectorized batching** and aggressive memory clearing (`torch.cuda.empty_cache()`) between model runs.
2. **Cross-Platform Compatibility:** We developed a custom **monkeypatch for Google's TimesFM** (see `src/models/timesfm.py`) to handle specific initialization bugs when running on diverse hardware (MPS/CUDA/CPU).
3. **Rolling-Origin Simulation:** The benchmarking uses a strict **rolling-origin backtest** across 80+ weeks, ensuring no temporal leakage and providing a realistic estimate of real-world performance.

### 🔬 Final Academic Takeaway
The "Italian ILI Forecasting" problem is no longer limited to locally-trained statistical models. **Zero-Shot Foundation Models** (specifically Chronos and TimesFM) provide a robust, scalable alternative that captures seasonal peaks with high fidelity and well-calibrated uncertainty. For public health authorities, these models represent a significant step forward in building resilient forecasting systems that do not require decades of local historical data to be effective.

## Step 6: Exporting Full Results
Finally, you can download the full set of forecasts and metrics (in CSV format) for further analysis.

In [ ]:
from google.colab import files

# Zip all Phase 2 results
!zip -r thesis_results_biagio.zip results/phase2/

# Download to your local machine
files.download('thesis_results_biagio.zip')

print("\nThank you for reviewing my work, Professor!")